# 11 - Validate operational and analytical observability

Validates the application ledger and gold tables before building Power BI and Real-Time Dashboard artifacts. Use Fabric's table chart options on each displayed result. Fabric job-level concurrency and duration come from Workspace monitoring `ItemJobEventLogs`; this notebook validates the application-level view.

In [ ]:
DATABASE = ""
TABLE_PREFIX = "people_counter"
LOOKBACK_DAYS = 30
TARGET_VIDEO_HOURS = 200000.0
DEADLINE_DAYS = 30.0
CAMERA_ID = ""
LOCATION_ID = ""

In [ ]:
from datetime import datetime, timedelta, timezone
import re

from pyspark.sql import SparkSession, Window, functions as F


IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")
database = DATABASE.strip()
prefix = TABLE_PREFIX.strip()
if database and IDENTIFIER.fullmatch(database) is None:
    raise ValueError("DATABASE is not a valid identifier")
if IDENTIFIER.fullmatch(prefix) is None:
    raise ValueError("TABLE_PREFIX is not a valid identifier")
lookback_days = int(LOOKBACK_DAYS)
if lookback_days < 1 or float(TARGET_VIDEO_HOURS) <= 0 or float(DEADLINE_DAYS) <= 0:
    raise ValueError("Lookback, target hours, and deadline must be positive")


def table(suffix: str) -> str:
    value = f"{prefix}_{suffix}"
    return f"{database}.{value}" if database else value


spark_candidate = globals().get("spark")
if not isinstance(spark_candidate, SparkSession):
    raise RuntimeError("A Fabric Spark session is required")
spark_session = spark_candidate
spark_session.conf.set("spark.sql.session.timeZone", "UTC")
now = datetime.now(timezone.utc)
start = now - timedelta(days=lookback_days)
work = spark_session.table(table("video_work"))
attempts = spark_session.table(table("video_attempts"))
gold_flow = spark_session.table(table("gold_flow_hour"))
gold_video = spark_session.table(table("gold_video"))
operations = spark_session.table(table("gold_operations_hour"))
findings = spark_session.table(table("reconciliation_findings"))

status_counts = work.groupBy("status").agg(
    F.count("work_id").alias("jobs"),
    (F.sum(F.coalesce("duration_seconds", F.lit(0.0))) / 3600.0).alias("video_hours"),
).orderBy(F.col("jobs").desc())
display(status_counts)

active = work.where(F.col("status").isin("LEASED", "STAGING", "RUNNING", "WRITING")).select(
    "work_id",
    "status",
    "camera_id",
    "location_id",
    "lease_owner_attempt_id",
    "lease_acquired_at",
    "last_heartbeat_at",
    "lease_expires_at",
    ((F.unix_timestamp(F.lit(now)) - F.unix_timestamp("last_heartbeat_at")) / 60.0).alias("heartbeat_age_minutes"),
).orderBy(F.col("heartbeat_age_minutes").desc())
display(active)

queue_health = work.where(F.col("status").isin("QUEUED", "RETRY_WAIT")).agg(
    F.count("work_id").alias("queue_depth"),
    F.min("queued_at").alias("oldest_queued_at"),
    F.max((F.unix_timestamp(F.lit(now)) - F.unix_timestamp("queued_at")) / 60.0).alias("oldest_queue_age_minutes"),
)
display(queue_health)

attempt_health = attempts.where(F.col("claimed_at") >= F.lit(start)).groupBy("status", "error_category").agg(
    F.count("attempt_id").alias("attempts"),
    F.avg("processing_seconds").alias("average_processing_seconds"),
    F.expr("percentile_approx(processing_seconds, 0.95)").alias("p95_processing_seconds"),
).orderBy(F.col("attempts").desc())
display(attempt_health)

burn_down = (
    operations.where(F.col("hour_utc") >= F.lit(start))
    .orderBy("hour_utc")
    .withColumn("completed_video_hours", F.sum("video_hours_completed").over(Window.orderBy("hour_utc")))
    .withColumn("remaining_video_hours", F.greatest(F.lit(0.0), F.lit(float(TARGET_VIDEO_HOURS)) - F.col("completed_video_hours")))
)
display(burn_down)

if CAMERA_ID:
    gold_flow = gold_flow.where(F.col("camera_id") == CAMERA_ID)
    gold_video = gold_video.where(F.col("camera_id") == CAMERA_ID)
if LOCATION_ID:
    gold_flow = gold_flow.where(F.col("location_id") == LOCATION_ID)
    gold_video = gold_video.where(F.col("location_id") == LOCATION_ID)

display(gold_flow.where(F.col("hour_utc") >= F.lit(start)).orderBy("hour_utc", "camera_id"))
display(
    gold_video.where(F.col("completed_at") >= F.lit(start)).groupBy("camera_id", "location_id").agg(
        F.count("work_id").alias("videos"),
        F.sum("line_in_count").alias("entries"),
        F.sum("line_out_count").alias("exits"),
        F.avg("distinct_people").alias("average_tracks_per_video"),
        F.avg("speed_x_realtime").alias("average_speed_x_realtime"),
    ).orderBy(F.col("entries").desc())
)
display(findings.where(F.col("resolved_at").isNull()).orderBy(F.col("severity"), F.col("detected_at")))

## Suggested Fabric chart mappings

- `status_counts`: donut chart, legend=`status`, value=`jobs`.
- `active`: table with conditional formatting on `heartbeat_age_minutes`.
- `queue_health`: cards for depth and oldest age.
- `burn_down`: line chart, x=`hour_utc`, y=`completed_video_hours` and `remaining_video_hours`.
- `gold_flow`: clustered column or line chart, x=`hour_utc`, y=`entries` and `exits`, series=`camera_id`.
- camera summary: bars for entries/exits and a card for average processing speed.

Recreate these as governed Power BI visuals over the Direct Lake semantic model. Use a Real-Time Dashboard over Workspace monitoring for Fabric `Not started`/`In progress` job counts; application `RUNNING` and Fabric job status are related but not interchangeable.